# ASISTENTE DE IA CON LLAMA 3

LLAMA 3 ES UN CONJUNTO DE MODELOS DE INTELIGENCIA ARTIFICIAL DISEÑADO POR META QUE SON DE CODIGO ABIERTO , VIENEN ENTRENADOS A FULL Y SOLO SE CALIBRAN PARA UNA TAREA EN ESPECIFICO.

# INICIEMOS EL MODELADO

In [1]:
!pip install -q transformers accelerate bitsandbytes gTTS gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 54.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 1.16.1 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [2]:
from huggingface_hub import login

login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# YA LOGEADOS DESCARGUEMOS EL MODELO LLAMA 3 PARA EMPEZAR OPERACIONES

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Versión pública sin bloqueo de formulario
model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"

print("⏳ Cargando Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("⏳ Cargando Llama 3 optimizado (tardará ~1 minuto)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("✅ ¡Llama 3 listo y cargado en la GPU sin restricciones!")

⏳ Cargando Tokenizer...


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

✅ ¡Llama 3 listo y cargado en la GPU sin restricciones!


# HAGAMOS QUE HABLE

In [7]:
import os
from gtts import gTTS
from IPython.display import Audio, display

# 1. Definir la función para generar texto con Llama 3
# 1. Definir la función corregida para generar texto con Llama 3
def generar_respuesta(prompt_usuario):
    messages = [
        {"role": "system", "content": "Eres un asistente de voz útil, amable y conciso. Responde en español y de forma breve (máximo 2 oraciones) para ser fluido en audio."},
        {"role": "user", "content": prompt_usuario}
    ]

    # Aplicar el template y retornar un diccionario de tensores PyTorch
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Mover todo el diccionario a la GPU
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extraer únicamente los tokens generados de respuesta
    input_length = inputs["input_ids"].shape[1]
    respuesta_texto = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return respuesta_texto.strip()

# 2. Definir la función de Texto a Voz (TTS)
def texto_a_audio(texto):
    tts = gTTS(text=texto, lang='es', slow=False)
    archivo_audio = "respuesta.mp3"
    tts.save(archivo_audio)
    return archivo_audio

# 3. Prueba rápida en texto y voz
pregunta = "¡Hola Llama 3! Preséntate brevemente y dime si estás listo para trabajar hoy."
print(f"👤 Usuario: {pregunta}\n")

print("🤖 Llama 3 generando respuesta...")
respuesta = generar_respuesta(pregunta)
print(f"🤖 Respuesta: {respuesta}\n")

print("🔊 Generando audio...")
audio_path = texto_a_audio(respuesta)
display(Audio(audio_path, autoplay=True))

👤 Usuario: ¡Hola Llama 3! Preséntate brevemente y dime si estás listo para trabajar hoy.

🤖 Llama 3 generando respuesta...


[transformers] Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


🤖 Respuesta: Hola! Soy Llama 3, un asistente de voz útil y amable. Estoy listo para ayudarte en lo que necesites, ¡pronto y concisamente!

🔊 Generando audio...


# Interfaz de respuesta

In [ ]:
import gradio as gr
import torch
import transformers

# 1. Cargar Whisper Small configurado explícitamente para ESPAÑOL
print("⏳ Cargando Whisper Small (Mayor precisión en español)...")
stt_pipeline = transformers.pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if torch.cuda.is_available() else -1
)

# 2. Función del pipeline con lenguaje forzado y respuestas ultra cortas
def procesar_asistente_voz(audio_path):
    if audio_path is None:
        return "Por favor, graba un audio para comenzar.", None

    # A. Escuchar (Forzando idioma Español)
    resultado_stt = stt_pipeline(
        audio_path,
        generate_kwargs={"language": "spanish", "task": "transcribe"}
    )
    transcripcion = resultado_stt["text"]

    # B. Razonar con Llama 3
    respuesta_texto = generar_respuesta(transcripcion)

    # C. Generar Audio
    respuesta_audio = texto_a_audio(respuesta_texto)

    return f"🗣️ Tú dijiste: {transcripcion}\n\n🤖 Llama 3: {respuesta_texto}", respuesta_audio

# 3. Interfaz de Gradio
demo = gr.Interface(
    fn=procesar_asistente_voz,
    inputs=gr.Audio(sources=["microphone"], type="filepath", label="🎙️ Haz tu pregunta"),
    outputs=[
        gr.Textbox(label="Transcripción y Respuesta"),
        gr.Audio(label="Respuesta de Voz", autoplay=True)
    ],
    title="🎙️ Asistente de Voz con Llama 3 (Español Optimizado)"
)

demo.launch(share=True, debug=True)

⏳ Cargando Whisper Small (Mayor precisión en español)...


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9d8edee1f47b7babaf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging